# C4 — Distance on Feature Space [Optional]

---

## 🔍 Problem-এ কী চাওয়া হয়েছে?

**C-Data-1** থেকে দুটো feature নেওয়া হবে: `Hours_Study` ও `GPA`

| ID | Hours_Study | GPA |
|---|---|---|
| 1 | 1.0 | 3.10 |
| 4 | 5.0 | 3.90 |

তিনটি কাজ করতে হবে:

| Task | কী করতে হবে |
|---|---|
| **(a)** | ID 1 ও ID 4-এর মধ্যে **Euclidean distance** হিসাব করা |
| **(b)** | একই pair-এর **Manhattan distance** হিসাব করা |
| **(c)** | `Hours_Study` ও `GPA`-কে **Min-Max Normalize** করে দুটো distance আবার হিসাব করা — তারপর **এক লাইনে** scale effect comment করা |


---

## 🎯 এই কাজ থেকে আমরা কী অর্জন করতে পারব?

- Real dataset-এর দুটো row-কে **feature vector** হিসেবে treat করতে শিখব।
- **Unscaled feature-এ** `Hours_Study` (0–5 range) ও `GPA` (2.3–3.9 range) distance-কে কীভাবে **আলাদাভাবে প্রভাবিত** করে দেখব।
- **Scaling-এর পরে** distance কীভাবে পরিবর্তিত হয় এবং কোন feature আগে distance dominate করছিল সেটা বুঝব।
- এই পুরো experiment ML-এ **KNN algorithm-এর আগে scaling কেন দরকার** সেটার সরাসরি প্রমাণ।


---

## 🧠 কীভাবে চিন্তা করতে হবে?

### Feature Vector হিসেবে চিন্তা করি:

ID 1 = point $(1.0, 3.10)$, ID 4 = point $(5.0, 3.90)$ — 2D feature space-এ দুটো point।

$$\text{Euclidean: } d_E = \sqrt{(5.0-1.0)^2 + (3.90-3.10)^2} = \sqrt{16 + 0.64} = \sqrt{16.64} \approx 4.079$$

$$\text{Manhattan: } d_M = |5.0-1.0| + |3.90-3.10| = 4.0 + 0.8 = 4.8$$

**লক্ষ্য করো:** `Hours_Study`-এর পার্থক্য = 4.0, `GPA`-এর পার্থক্য = 0.8।
`Hours_Study` distance-এ **5× বেশি** অবদান রাখছে শুধু scale-এর কারণে!

### Scaling-এর পরে কী হবে?

Min-Max করলে উভয় feature [0, 1]-এ আসবে → `Hours_Study` ও `GPA` **সমান গুরুত্ব** পাবে।
Distance অনেক ছোট হবে কারণ values এখন 0–1 range-এ।


---

## 🛠️ Problem Solve করার Approach

**Step 1:** C-Data-1 তৈরি করে ID 1 ও ID 4 extract করা।

**Step 2 (Task a):** Euclidean distance হাতে হিসাব + code verify।

**Step 3 (Task b):** Manhattan distance হাতে হিসাব + code verify।

**Step 4 (Task c):** সব ৫টি row Min-Max scale করে ID 1 ও ID 4 extract, তারপর দুটো distance আবার হিসাব করা।

**Step 5:** Before vs After তুলনা করে এক লাইনে comment।


## Step 1: C-Data-1 তৈরি ও ID 1, ID 4 Extract করা

In [10]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Now we'll create C-Data-1 and extract the two relevant rows
data1 = pd.DataFrame({
    'ID':           [1, 2, 3, 4, 5],
    'Age':          [20, 21, 22, 20, 23],
    'Hours_Study':  [1.0, 0.5, 2.2, 5.0, 0.2],
    'GPA':          [3.10, 2.60, 3.40, 3.90, 2.30],
    'Internet':     ['Yes', 'No', 'Yes', 'Yes', 'No'],
    'City':         ['Dhaka', 'Chattogram', 'Rajshahi', 'Dhaka', 'Rajshahi']
})
# Here we've created the full C-Data-1 dataset.

id1 = data1.loc[data1['ID'] == 1, ['Hours_Study', 'GPA']].values.flatten()
id4 = data1.loc[data1['ID'] == 4, ['Hours_Study', 'GPA']].values.flatten()
# Here we've extracted the feature vectors for ID 1 and ID 4.

print("C-Data-1 (Hours_Study & GPA):")
print(data1[['ID', 'Hours_Study', 'GPA']].to_string(index=False))
print()
print(f"ID 1 feature vector : {id1}  (Hours_Study=1.0, GPA=3.10)")
print(f"ID 4 feature vector : {id4}  (Hours_Study=5.0, GPA=3.90)")


C-Data-1 (Hours_Study & GPA):
 ID  Hours_Study  GPA
  1          1.0  3.1
  2          0.5  2.6
  3          2.2  3.4
  4          5.0  3.9
  5          0.2  2.3

ID 1 feature vector : [1.  3.1]  (Hours_Study=1.0, GPA=3.10)
ID 4 feature vector : [5.  3.9]  (Hours_Study=5.0, GPA=3.90)


`data1.loc[data1['ID'] == 1, ['Hours_Study', 'GPA']]` → শর্ত দিয়ে নির্দিষ্ট row ও column select।
`.values.flatten()` → DataFrame থেকে NumPy array-এ রূপান্তর, 1D করা।


---

## Task (a): Euclidean Distance — হাতের হিসাব

$$d_E = \sqrt{(\text{Hours}_4 - \text{Hours}_1)^2 + (\text{GPA}_4 - \text{GPA}_1)^2}$$

$$= \sqrt{(5.0 - 1.0)^2 + (3.90 - 3.10)^2} = \sqrt{16 + 0.64} = \sqrt{16.64} \approx \mathbf{4.079}$$

**লক্ষ্য করো:** Hours contribution = 16, GPA contribution = 0.64 → Hours **25×** বেশি contribute করছে!


In [11]:
# Now we'll compute Euclidean distance step by step
diff       = id4 - id1
sq_diff    = diff ** 2
euc_dist   = np.sqrt(sq_diff.sum())
# Here we've computed element-wise difference, squared, summed, and square-rooted.

print("Euclidean Distance — Step by Step:")
print(f"  Difference (id4 - id1)     : {diff}")
print(f"  Squared differences        : {sq_diff}")
print(f"  Hours contribution         : {sq_diff[0]:.4f}  ({sq_diff[0]/sq_diff.sum()*100:.1f}% of total)")
print(f"  GPA   contribution         : {sq_diff[1]:.4f}  ({sq_diff[1]/sq_diff.sum()*100:.1f}% of total)")
print(f"  Sum of squares             : {sq_diff.sum():.4f}")
print(f"  Euclidean distance         : {euc_dist:.4f}")


Euclidean Distance — Step by Step:
  Difference (id4 - id1)     : [4.  0.8]
  Squared differences        : [16.    0.64]
  Hours contribution         : 16.0000  (96.2% of total)
  GPA   contribution         : 0.6400  (3.8% of total)
  Sum of squares             : 16.6400
  Euclidean distance         : 4.0792


`diff ** 2` → element-wise বর্গ।
`sq_diff.sum()` → বর্গের যোগফল।
`np.sqrt()` → বর্গমূল।

**Contribution percentage দেখো:** Hours_Study = 96.1%, GPA = 3.9% — GPA প্রায় invisible!
এটাই unscaled distance-এর সমস্যা।


---

## Task (b): Manhattan Distance — হাতের হিসাব

$$d_M = |\text{Hours}_4 - \text{Hours}_1| + |\text{GPA}_4 - \text{GPA}_1| = |4.0| + |0.8| = \mathbf{4.8}$$


In [12]:
# Now we'll compute Manhattan distance
abs_diff   = np.abs(id4 - id1)
man_dist   = abs_diff.sum()
# Here we've taken absolute differences and summed them.

print("Manhattan Distance — Step by Step:")
print(f"  Absolute differences       : {abs_diff}")
print(f"  Hours contribution         : {abs_diff[0]:.4f}  ({abs_diff[0]/abs_diff.sum()*100:.1f}% of total)")
print(f"  GPA   contribution         : {abs_diff[1]:.4f}  ({abs_diff[1]/abs_diff.sum()*100:.1f}% of total)")
print(f"  Manhattan distance         : {man_dist:.4f}")


Manhattan Distance — Step by Step:
  Absolute differences       : [4.  0.8]
  Hours contribution         : 4.0000  (83.3% of total)
  GPA   contribution         : 0.8000  (16.7% of total)
  Manhattan distance         : 4.8000


`np.abs()` → absolute value (ঋণাত্মক মান থাকলেও ধনাত্মক হয়)।
`.sum()` → সব absolute difference যোগ করা।

**Hours_Study = 83.3%, GPA = 16.7%** — Manhattan-এও Hours dominates।


---

## Task (c): Min-Max Normalize করে Distance Recompute করা

সম্পূর্ণ dataset-এর উপর Min-Max fitting করতে হবে — শুধু ID 1 ও 4-এর উপর নয়।
কারণ: scaler-কে সব data দেখতে হয় (min ও max বের করতে)।


In [4]:
# Now we'll Min-Max scale Hours_Study and GPA using the full dataset
mm = MinMaxScaler()
scaled_features = mm.fit_transform(data1[['Hours_Study', 'GPA']])
# Here we've fitted the scaler on all 5 rows and transformed both features.

scaled_df = pd.DataFrame(scaled_features,
                         columns=['Hours_Scaled', 'GPA_Scaled'])
scaled_df['ID'] = data1['ID'].values

print("Min-Max Scaled Features (all 5 rows):")
print(scaled_df[['ID', 'Hours_Scaled', 'GPA_Scaled']].round(4).to_string(index=False))


Min-Max Scaled Features (all 5 rows):
 ID  Hours_Scaled  GPA_Scaled
  1        0.1667      0.5000
  2        0.0625      0.1875
  3        0.4167      0.6875
  4        1.0000      1.0000
  5        0.0000      0.0000


`mm.fit_transform(data1[['Hours_Study', 'GPA']])` → scaler সব ৫টি row দেখে min/max শিখে, তারপর scale করে।
শুধু ID 1 ও 4-এর উপর fit করলে ভুল scaling হতো।


In [13]:
# Now we'll extract scaled vectors for ID 1 and ID 4
id1_scaled = scaled_df.loc[scaled_df['ID'] == 1,
                            ['Hours_Scaled', 'GPA_Scaled']].values.flatten()
id4_scaled = scaled_df.loc[scaled_df['ID'] == 4,
                            ['Hours_Scaled', 'GPA_Scaled']].values.flatten()
# Here we've extracted the scaled feature vectors for the two IDs.

print(f"ID 1 scaled : {id1_scaled.round(4)}")
print(f"ID 4 scaled : {id4_scaled.round(4)}")


ID 1 scaled : [0.1667 0.5   ]
ID 4 scaled : [1. 1.]


Scaled vector extract করা হয়েছে।
এখন উভয় feature [0, 1]-এ — Hours ও GPA সমান playing field-এ।


In [14]:
# Now we'll recompute both distances on the scaled vectors
diff_s      = id4_scaled - id1_scaled
sq_diff_s   = diff_s ** 2
euc_scaled  = np.sqrt(sq_diff_s.sum())

abs_diff_s  = np.abs(diff_s)
man_scaled  = abs_diff_s.sum()
# Here we've computed Euclidean and Manhattan distances on the normalized features.

print("Scaled Euclidean — Step by Step:")
print(f"  Scaled difference          : {diff_s.round(4)}")
print(f"  Hours contribution         : {sq_diff_s[0]:.4f}  ({sq_diff_s[0]/sq_diff_s.sum()*100:.1f}%)")
print(f"  GPA   contribution         : {sq_diff_s[1]:.4f}  ({sq_diff_s[1]/sq_diff_s.sum()*100:.1f}%)")
print(f"  Euclidean distance (scaled): {euc_scaled:.4f}")
print()
print("Scaled Manhattan — Step by Step:")
print(f"  Absolute scaled diff       : {abs_diff_s.round(4)}")
print(f"  Hours contribution         : {abs_diff_s[0]:.4f}  ({abs_diff_s[0]/abs_diff_s.sum()*100:.1f}%)")
print(f"  GPA   contribution         : {abs_diff_s[1]:.4f}  ({abs_diff_s[1]/abs_diff_s.sum()*100:.1f}%)")
print(f"  Manhattan distance (scaled): {man_scaled:.4f}")


Scaled Euclidean — Step by Step:
  Scaled difference          : [0.8333 0.5   ]
  Hours contribution         : 0.6944  (73.5%)
  GPA   contribution         : 0.2500  (26.5%)
  Euclidean distance (scaled): 0.9718

Scaled Manhattan — Step by Step:
  Absolute scaled diff       : [0.8333 0.5   ]
  Hours contribution         : 0.8333  (62.5%)
  GPA   contribution         : 0.5000  (37.5%)
  Manhattan distance (scaled): 1.3333


Scaling-এর পরে contribution percentage দেখো — Hours ও GPA এখন অনেক কাছাকাছি।


## Step 5: Before vs After — সম্পূর্ণ তুলনা

In [15]:
# Now we'll build a final comparison summary
summary = pd.DataFrame({
    'Metric':     ['Euclidean', 'Manhattan'],
    'Unscaled':   [round(euc_dist, 4),   round(man_dist, 4)],
    'Scaled':     [round(euc_scaled, 4), round(man_scaled, 4)],
    'Change':     [f'{euc_scaled/euc_dist:.2f}x smaller',
                   f'{man_scaled/man_dist:.2f}x smaller']
})
# Here we've summarised the distance before and after scaling side by side.

print("── Before vs After Min-Max Scaling ──")
print(summary.to_string(index=False))
print()

# Now we'll show the contribution breakdown before vs after
print("Contribution breakdown (Hours vs GPA):")
print(f"  BEFORE — Euclidean: Hours={sq_diff[0]/sq_diff.sum()*100:.1f}%,  GPA={sq_diff[1]/sq_diff.sum()*100:.1f}%")
print(f"  AFTER  — Euclidean: Hours={sq_diff_s[0]/sq_diff_s.sum()*100:.1f}%, GPA={sq_diff_s[1]/sq_diff_s.sum()*100:.1f}%")
print()
print(f"  BEFORE — Manhattan: Hours={abs_diff[0]/abs_diff.sum()*100:.1f}%,  GPA={abs_diff[1]/abs_diff.sum()*100:.1f}%")
print(f"  AFTER  — Manhattan: Hours={abs_diff_s[0]/abs_diff_s.sum()*100:.1f}%, GPA={abs_diff_s[1]/abs_diff_s.sum()*100:.1f}%")


── Before vs After Min-Max Scaling ──
   Metric  Unscaled  Scaled        Change
Euclidean    4.0792  0.9718 0.24x smaller
Manhattan    4.8000  1.3333 0.28x smaller

Contribution breakdown (Hours vs GPA):
  BEFORE — Euclidean: Hours=96.2%,  GPA=3.8%
  AFTER  — Euclidean: Hours=73.5%, GPA=26.5%

  BEFORE — Manhattan: Hours=83.3%,  GPA=16.7%
  AFTER  — Manhattan: Hours=62.5%, GPA=37.5%


Contribution percentage-এর পরিবর্তনটাই সবচেয়ে গুরুত্বপূর্ণ।
**Before scaling:** GPA প্রায় ignored (3.9%)
**After scaling:** উভয় feature ন্যায্য অংশ পাচ্ছে।


---

## ✅ এক লাইনে Scale Effect Comment

> **Unscaled-এ `Hours_Study` (range 0–5) distance-কে ~96% dominate করে GPA (range 2.3–3.9)-কে প্রায় invisible করে দেয়, কিন্তু Min-Max scaling-এর পরে উভয় feature [0,1]-এ আসায় তাদের contribution balanced হয় এবং distance অনেক ছোট ও অর্থপূর্ণ হয়।**


In [17]:
# Now we'll print the one-line scale effect as a formatted statement
print("=" * 60)
print("  ONE-LINE SCALE EFFECT COMMENT")
print("=" * 60)
print()
print(f"  Unscaled Euclidean : {euc_dist:.4f}")
print(f"  Scaled   Euclidean : {euc_scaled:.4f}  ({euc_scaled/euc_dist*100:.1f}% of original)")
print()
print("  Before: Hours dominates (96.1%), GPA nearly invisible (3.9%)")
print("  After : both features contribute fairly — balanced distance")
print()
print("  Conclusion: Always scale before using distance-based models.")
# Here we've printed the final verdict of the scaling experiment.


  ONE-LINE SCALE EFFECT COMMENT

  Unscaled Euclidean : 4.0792
  Scaled   Euclidean : 0.9718  (23.8% of original)

  Before: Hours dominates (96.1%), GPA nearly invisible (3.9%)
  After : both features contribute fairly — balanced distance

  Conclusion: Always scale before using distance-based models.


এই experiment-ই KNN, K-Means-এর আগে **feature scaling কেন বাধ্যতামূলক** সেটার সরাসরি প্রমাণ।
শুধু range-এর পার্থক্যের কারণে একটি feature অন্যটিকে **সম্পূর্ণ suppress** করতে পারে।


---